# BoardGameGeek recommender

Mourad Kraiem and Mohamed Oussama Zoghlami · ENSI

Recommend games from a rating profile or up to three games a player likes.

Install `requirements.txt`, place both BoardGameGeek CSVs in `data/`, and run the notebook from top to bottom. The [project review](PROJECT_REVIEW.md) maps the work to the seven chapters.

In [1]:
from os import environ
from pathlib import Path
from ast import literal_eval
from collections import defaultdict
from html import unescape
import re

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.linalg import solve
from scipy.sparse import csr_matrix, vstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

SEED, K = 42, 10
USER_SAMPLE_PERCENT, MIN_RATINGS, LIKE = 2, 10, 8
EASE_SAMPLE_PERCENT = 50


# Run from project/ or from its parent folder.
project_dir = Path("project") if Path("project/requirements.txt").is_file() else Path(".")
DATA_DIR = Path(environ.get("BGG_DATA_DIR", project_dir / "data"))
for filename in ("games_detailed_info.csv", "bgg-19m-reviews.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"Put {filename} in {DATA_DIR.resolve()}.")

## 1. Load and clean the ratings

Use 2% of user histories for the original benchmark. Keep base games, average duplicate ratings, and require at least 10 ratings per user and game. Save extra histories for EASE.

In [2]:
limits = ["minplayers", "maxplayers", "maxplaytime", "averageweight"]
fields = ["id", "type", "primary", "description", "boardgamecategory", "boardgamemechanic"]
games = pd.read_csv(f"{DATA_DIR}/games_detailed_info.csv", usecols=fields + limits)
games = games[games["type"] == "boardgame"].sort_values("id").reset_index(drop=True)
games = games.rename(columns={"id": "game_id", "primary": "title"})
game_ids = pd.Index(games["game_id"])
games[limits] = games[limits].where(games[limits] > 0)

chunks, extra_chunks = [], []
raw_rating_count = 0
for chunk in pd.read_csv(f"{DATA_DIR}/bgg-19m-reviews.csv", usecols=["user", "ID", "rating"],
                         dtype={"user": "string"}, chunksize=300_000):
    raw_rating_count += len(chunk)
    user_hash = pd.util.hash_pandas_object(chunk["user"], index=False)
    bucket = user_hash % 100
    valid = chunk["user"].notna() & chunk["ID"].isin(game_ids) & chunk["rating"].between(1, 10)
    chunks.append(chunk[valid & (bucket < USER_SAMPLE_PERCENT)])
    extra_rows = valid & bucket.between(USER_SAMPLE_PERCENT, EASE_SAMPLE_PERCENT - 1)
    extra_chunks.append(chunk.loc[extra_rows, ["ID", "rating"]].assign(user=user_hash[extra_rows]))

ratings = pd.concat(chunks).groupby(["user", "ID"], as_index=False)["rating"].mean()
del chunks
while True:
    user_count = ratings.groupby("user")["rating"].transform("size")
    game_count = ratings.groupby("ID")["rating"].transform("size")
    keep = (user_count >= MIN_RATINGS) & (game_count >= MIN_RATINGS)
    if keep.all():
        break
    ratings = ratings[keep]

ratings = ratings.reset_index(drop=True)
ratings["user_id"] = pd.factorize(ratings.pop("user"), sort=True)[0]
ratings["item"] = game_ids.get_indexer(ratings["ID"])
n_users, n_items = ratings["user_id"].nunique(), ratings["item"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_items)
print(f"{raw_rating_count:,} raw rating rows | {len(games):,} base-game metadata records")
print(f"{len(ratings):,} ratings | {n_users:,} users | {n_items:,} games | {sparsity:.2%} sparse")
display(ratings["rating"].describe().round(2), games[["title", *limits]].head())

18,964,807 raw rating rows | 21,631 base-game metadata records
324,348 ratings | 4,365 users | 4,709 games | 98.42% sparse


count    324348.00
mean          7.15
std           1.52
min           1.00
25%           6.00
50%           7.00
75%           8.00
max          10.00

,title,minplayers,maxplayers,maxplaytime,averageweight
0,Die Macher,3.0,5.0,240.0,4.3206
1,Dragonmaster,3.0,4.0,30.0,1.9630
2,Samurai,2.0,4.0,60.0,2.4866
3,Tal der Könige,2.0,4.0,60.0,2.6667
4,Acquire,2.0,6.0,90.0,2.5019


## 2. Set aside likes for validation and testing

Hide two ratings of 8 or higher for each evaluated user. Keep at least one training rating per game.

In [3]:
shuffled = ratings.sample(frac=1, random_state=SEED)
reserved = shuffled.drop_duplicates("item").index
liked = shuffled[shuffled["rating"] >= LIKE]
pool = liked.drop(index=reserved, errors="ignore")

liked_count = liked.groupby("user_id").size()
pool_count = pool.groupby("user_id").size()
eligible = liked_count[liked_count >= 3].index.intersection(pool_count[pool_count >= 2].index)
selected = np.random.default_rng(SEED).permutation(eligible.to_numpy())[:700]

held_out = pool[pool["user_id"].isin(selected)].groupby("user_id").head(2)
train = ratings.drop(held_out.index)
targets = held_out.groupby("user_id")["item"].agg(list)
n_validation = round(len(selected) * 2 / 7)
validation, test = targets.loc[selected[:n_validation]], targets.loc[selected[n_validation:]]
validation, test = validation.sort_index(), test.sort_index()
warm_items = np.sort(train["item"].unique())
print(f"Training: {len(train):,} ratings | Validation: {len(validation)} users | Test: {len(test)} users")

Training: 322,948 ratings | Validation: 200 users | Test: 500 users


## 3. Build and compare the baselines

Compare popularity, User–User, Item–Item, SVD, and TF-IDF. Validation selects the weight for combining collaborative scores with content.

In [4]:
history = csr_matrix((train["rating"], (train["user_id"], train["item"])),
                     shape=(n_users, len(games)))
likes = history.copy()
likes.data = (likes.data >= LIKE).astype(np.float32)
likes.eliminate_zeros()
popularity = np.asarray(likes.sum(axis=0)).ravel()
users = normalize(likes, axis=1)
items = normalize(likes, axis=0).tocsr()
users_t, items_t = users.T.tocsr(), items.T.tocsr()

svd = TruncatedSVD(n_components=32, random_state=SEED)
user_factors = svd.fit_transform(likes)


def tags(value, prefix):
    return " ".join(prefix + re.sub(r"[^\w]+", "_", tag.lower()) for tag in literal_eval(value))


categories = games["boardgamecategory"].fillna("[]").map(lambda value: tags(value, "cat_"))
mechanics = games["boardgamemechanic"].fillna("[]").map(lambda value: tags(value, "mec_"))
text = games["description"].fillna("").map(unescape) + " " + (categories + " " + mechanics + " ") * 2
features = TfidfVectorizer(stop_words="english", strip_accents="unicode",
                           max_features=20_000, dtype=np.float32).fit_transform(text)
features_t = features.T.tocsr()


def content_scores(game_rows, weights):
    profile = normalize(csr_matrix(weights.reshape(1, -1)) @ features[game_rows])
    return (profile @ features_t).toarray().ravel()


def liked_profile(user):
    profile = history[user]
    profile.data = np.where(profile.data >= LIKE, profile.data - LIKE + 1, 0)
    profile.eliminate_zeros()
    return profile


def model_scores(user, candidates):
    profile = liked_profile(user)

    similarity = (users[user] @ users_t).toarray().ravel()
    similarity[user] = 0
    neighbors = np.argsort(-similarity, kind="stable")[:40]
    scores = {
        "Popularity": popularity,
        "User–User": likes[neighbors].T @ similarity[neighbors],
        "Item–Item": (profile @ items_t @ items).toarray().ravel(),
        "SVD": user_factors[user] @ svd.components_,
        "Content": content_scores(profile.indices, profile.data),
    }
    return {name: score[candidates] for name, score in scores.items()}


def scale(scores):
    scores = scores - scores.min()
    return scores / (scores.max() or 1)


def top_items(scores, candidates, k=K):
    order = np.lexsort((candidates, -popularity[candidates], -scores))
    return candidates[order[:k]]

In [5]:
CF_MODELS = ["User–User", "Item–Item", "SVD"]
METRICS = ["HitRate", "Precision", "Recall", "MRR", "NDCG", "Coverage"]
discount = 1 / np.log2(np.arange(2, K + 2))


def evaluate(targets, hybrids=None, score_fn=model_scores):
    metrics = defaultdict(list)
    covered = defaultdict(set)
    for user, relevant in targets.items():
        candidates = warm_items[~np.isin(warm_items, history[user].indices)]
        scores = score_fn(user, candidates)
        if hybrids:
            normalized = {name: scale(scores[name]) for name in [*CF_MODELS, "Content"]}
            for name, (cf, weight) in hybrids.items():
                scores[name] = weight * normalized[cf] + (1 - weight) * normalized["Content"]

        for name, values in scores.items():
            recommended = top_items(values, candidates)
            covered[name].update(recommended)
            hits = np.flatnonzero(np.isin(recommended, relevant))
            count = len(hits)
            metrics[name].append([
                count > 0, count / K, count / len(relevant),
                1 / (hits[0] + 1) if count else 0,
                discount[hits].sum() / discount[:len(relevant)].sum(),
            ])

    names = list(metrics)
    rows = [list(np.mean(metrics[name], axis=0)) + [len(covered[name]) / len(warm_items)]
            for name in names]
    return pd.DataFrame(rows, index=names, columns=[f"{name}@{K}" for name in METRICS])


grid = {f"{name} + content ({weight:.0%} CF)": (name, weight)
        for name in CF_MODELS for weight in [0.25, 0.50, 0.75]}
metric = f"NDCG@{K}"
val_scores = evaluate(validation, grid)[metric].sort_values(ascending=False)
best = val_scores.loc[list(grid)].idxmax()
cf_name, cf_weight = grid[best]
print(f"Selected hybrid: {best}")
display(val_scores.round(4).to_frame())
results = evaluate(test, {"Hybrid": (cf_name, cf_weight)}).sort_values(metric, ascending=False)
display(results.round(4))

Selected hybrid: User–User + content (50% CF)


,NDCG@10
User–User + content (50% CF),0.1684
User–User + content (75% CF),0.1671
User–User,0.1622
Item–Item + content (75% CF),0.1594
Item–Item + content (50% CF),0.1565
Item–Item,0.1555
SVD + content (75% CF),0.1326
SVD,0.1248
SVD + content (50% CF),0.1190
User–User + content (25% CF),0.1017


,HitRate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Coverage@10
Hybrid,0.356,0.0394,0.197,0.1682,0.1396,0.1225
User–User,0.356,0.0400,0.200,0.1627,0.1379,0.0548
Item–Item,0.312,0.0348,0.174,0.1215,0.1096,0.0862
SVD,0.294,0.0332,0.166,0.1219,0.1081,0.0542
Popularity,0.248,0.0272,0.136,0.0938,0.0846,0.0130
Content,0.084,0.0088,0.044,0.0424,0.0332,0.2081


## 4. Fit EASE and compare it with the hybrid

Fit EASE on 50% of user buckets. It learns how other ratings help predict the score of each game. Validation selects how much TF-IDF to add.

In [6]:
extra = pd.concat(extra_chunks, ignore_index=True)
del extra_chunks
extra = extra[extra["ID"].isin(game_ids[warm_items])]
extra = extra.groupby(["user", "ID"], sort=False, as_index=False)["rating"].mean()
extra = extra[extra.groupby("user")["rating"].transform("size") >= MIN_RATINGS]
extra_users = pd.factorize(extra["user"])[0]
extra_items = pd.Index(game_ids[warm_items]).get_indexer(extra["ID"])
more_ratings = csr_matrix(
    (extra["rating"].to_numpy(dtype=np.float32), (extra_users, extra_items)),
    shape=(extra["user"].nunique(), len(warm_items)),
)
del extra

ease_train = vstack([history[:, warm_items].astype(np.float32), more_ratings], format="csr")
print(f"EASE training: {ease_train.nnz:,} ratings, {ease_train.shape[0]:,} users")

ease_train.data = (ease_train.data / 10) ** 3
ease_profiles = ease_train[:n_users]
with threadpool_limits(limits=4):
    gram = (ease_train.T @ ease_train).toarray()
    # Add regularization to the diagonal before solving.
    gram.flat[::len(gram) + 1] += 50
    precision = solve(gram, np.eye(len(gram), dtype=np.float32), assume_a="pos")
ease_weights = -precision / np.diag(precision)
np.fill_diagonal(ease_weights, 0)
del ease_train, more_ratings, gram, precision


def ease_scores(user, candidates):
    warm_scores = (ease_profiles[user] @ ease_weights).ravel()
    return warm_scores[np.searchsorted(warm_items, candidates)]


def ease_hybrid_scores(user, candidates):
    collaborative = ease_scores(user, candidates)
    profile = liked_profile(user)
    content = content_scores(profile.indices, profile.data)[candidates]
    scores = {"EASE": collaborative}
    for weight in (0.05, 0.10, 0.20, 0.35, 0.50):
        scores[f"EASE + {weight:.0%} content"] = (
            (1 - weight) * scale(collaborative) + weight * scale(content)
        )
    return scores


ease_validation = evaluate(validation, score_fn=ease_hybrid_scores)
selected_ease = ease_validation[metric].idxmax()
print(f"Selected on validation: {selected_ease}")
display(ease_validation[[metric]].sort_values(metric, ascending=False).round(4))


def selected_ease_scores(user, candidates):
    scores = ease_hybrid_scores(user, candidates)
    return {name: scores[name] for name in dict.fromkeys(["EASE", selected_ease])}


ease_test = evaluate(test, score_fn=selected_ease_scores)
comparison = pd.concat([results, ease_test]).sort_values(metric, ascending=False)
display(comparison.round(4))

EASE training: 8,076,953 ratings, 108,241 users
Selected on validation: EASE + 10% content


,NDCG@10
EASE + 10% content,0.2523
EASE,0.2483
EASE + 5% content,0.2469
EASE + 20% content,0.2447
EASE + 35% content,0.2357
EASE + 50% content,0.2037


,HitRate@10,Precision@10,Recall@10,MRR@10,NDCG@10,Coverage@10
EASE,0.500,0.0604,0.302,0.2500,0.2130,0.1493
EASE + 10% content,0.492,0.0590,0.295,0.2477,0.2099,0.1501
Hybrid,0.356,0.0394,0.197,0.1682,0.1396,0.1225
User–User,0.356,0.0400,0.200,0.1627,0.1379,0.0548
Item–Item,0.312,0.0348,0.174,0.1215,0.1096,0.0862
SVD,0.294,0.0332,0.166,0.1219,0.1081,0.0542
Popularity,0.248,0.0272,0.136,0.0938,0.0846,0.0130
Content,0.084,0.0088,0.044,0.0424,0.0332,0.2081


## 5. Results on fresh users

Each test held out 1,000 likes from 500 users. Hit rate measures whether a user's top 10 includes a hidden like. NDCG also rewards placing it higher.

**New users**

| Games entered | Model | Hit rate@10 | NDCG@10 | Coverage@10 | Likes found |
| ---: | --- | ---: | ---: | ---: | ---: |
| 0 | Popularity | 20.0% | 0.0623 | 0.06% | 110 |
| 1 | Item–Item | 18.8% | 0.0637 | 5.02% | 103 |
| 1 | Hybrid | 16.2% | 0.0563 | 6.26% | 91 |
| 1 | Hybrid + popularity | 21.8% | 0.0726 | 2.63% | 117 |
| 3 | Item–Item | 26.4% | 0.0957 | 2.41% | 152 |
| 3 | Hybrid | 25.6% | 0.0949 | 3.35% | 146 |
| 3 | Hybrid + popularity | 27.2% | 0.0970 | 1.24% | 152 |

The one-game and three-game tests use the same users, candidates, and hidden likes. The hybrid uses a 20% popularity weight selected on validation.

**Users with rating history**

| Model | User buckets | EASE penalty | Hit rate@10 | NDCG@10 | Likes found |
| --- | ---: | ---: | ---: | ---: | ---: |
| Popularity | — | — | 29.0% | 0.0998 | 159 |
| EASE | 50% | 50 | 53.4% | 0.2252 | 313 |
| EASE | 50% | 100 | 53.4% | 0.2257 | 314 |
| EASE | 100% | 50 | 52.8% | 0.2251 | 314 |
| EASE | 100% | 800 | 51.4% | 0.2227 | 306 |

Tuning recovered one more like at 50%. The demo uses the original EASE and TF-IDF blend.

## 6. Score games from a few likes

Combine Item–Item similarity, TF-IDF, and shared game attributes. Average the scores for the games the player enters, then add the validation-selected popularity weight.

In [7]:
def make_like_matrix(frame):
    """Keep users with enough ratings and record games they rated at least 8."""
    rating_counts = frame.groupby("user_id")["rating"].transform("size")
    frame = frame.loc[rating_counts >= MIN_RATINGS]
    user_rows, users = pd.factorize(frame["user_id"], sort=False)
    liked = frame["rating"].to_numpy() >= LIKE
    return csr_matrix(
        (np.ones(liked.sum(), dtype=np.int32),
         (user_rows[liked], frame["item"].to_numpy()[liked])),
        shape=(len(users), len(warm_items)),
    )


def fit_cold_likes():
    warm_game_ids = game_ids[warm_items]
    held_out_users = validation.index.union(test.index)
    core = ratings.loc[~ratings["user_id"].isin(held_out_users), ["user_id", "item", "rating"]].copy()
    core["item"] = np.searchsorted(warm_items, core["item"])
    blocks = [make_like_matrix(core)]

    # Keep entire user histories together and process smaller groups to limit memory.
    bucket_ranges = [(2, 26), (26, 50), (50, 74), (74, 98)]
    groups = [[] for _ in bucket_ranges]
    for chunk in pd.read_csv(DATA_DIR / "bgg-19m-reviews.csv", usecols=["user", "ID", "rating"],
                             dtype={"user": "string"}, chunksize=300_000):
        items = warm_game_ids.get_indexer(chunk["ID"])
        valid = (chunk["user"].notna() & chunk["rating"].between(1, 10) & (items >= 0)).to_numpy()
        hashes = pd.util.hash_pandas_object(chunk.loc[valid, "user"], index=False).to_numpy()
        buckets = hashes % 100
        items = items[valid].astype(np.int32)
        values = chunk.loc[valid, "rating"].to_numpy()
        for group, (low, high) in zip(groups, bucket_ranges):
            keep = (low <= buckets) & (buckets < high)
            group.append(pd.DataFrame({"user_id": hashes[keep], "item": items[keep], "rating": values[keep]}))

    # Buckets 98 and 99 are reserved for validation and testing.
    for group in groups:
        frame = pd.concat(group, ignore_index=True)
        group.clear()
        frame = frame.groupby(["user_id", "item"], as_index=False, sort=False)["rating"].mean()
        blocks.append(make_like_matrix(frame))
    return vstack(blocks, format="csr")


cold_likes = fit_cold_likes()
cold_likes_by_game = cold_likes.tocsc()
cold_popularity = np.zeros(len(games), dtype=np.int64)
cold_popularity[warm_items] = np.diff(cold_likes_by_game.indptr)
popularity_prior = np.log1p(cold_popularity)
popularity_prior /= popularity_prior.max() or 1
print(f"Cold-start model: {cold_likes.shape[0]:,} users and {cold_likes.nnz:,} likes")

Cold-start model: 211,230 users and 6,108,959 likes


In [8]:
mechanic_families = {
    "Card play": ("Hand Management", "Deck, Bag, and Pool Building", "Open Drafting",
                  "Closed Drafting", "Card Drafting", "Trick-taking"),
    "Economic decisions": ("Auction/Bidding", "Auction: English", "Auction: Dutch", "Income",
                           "Market", "Trading", "Worker Placement", "Worker Placement, Different Worker Types"),
    "Area control and movement": ("Area Majority / Influence", "Area Control / Area Influence", "Area Movement",
                                  "Area-Impulse", "Campaign / Battle Card Driven", "Grid Movement"),
    "Spatial construction": ("Tile Placement", "Pattern Building", "Grid Coverage", "Network and Route Building"),
    "Social deduction": ("Deduction", "Hidden Roles", "Hidden Traitor", "Voting", "Communication Limits"),
    "Narrative play": ("Narrative Choice / Paragraph", "Storytelling", "Role Playing",
                       "Scenario / Mission / Campaign Game"),
    "Shared goals": ("Cooperative Game", "Semi-Cooperative Game"),
}


def tag_key(value):
    return " ".join(re.findall(r"[^\W_]+", value.casefold()))


def game_tags(column):
    return [{tag_key(tag) for tag in literal_eval(value)} - {""} for value in games[column].fillna("[]")]


family_by_tag = {tag_key(tag): family for family, tags in mechanic_families.items() for tag in tags}
mechanic_tags = game_tags("boardgamemechanic")
category_tags = game_tags("boardgamecategory")
family_tags = [{family_by_tag[tag] for tag in tags if tag in family_by_tag} for tags in mechanic_tags]


def score_liked_game(seed):
    """Combine description, tag overlap, and likes from other players."""
    content = content_scores([seed], np.ones(1))
    knowledge = np.zeros(len(games))
    total_weight = 0.0
    for weight, tags in [(0.45, mechanic_tags), (0.25, category_tags), (0.30, family_tags)]:
        seed_tags = tags[seed]
        if seed_tags:
            # Jaccard similarity is the share of shared tags in the union.
            knowledge += weight * np.array([len(seed_tags & other) / len(seed_tags | other) for other in tags])
            total_weight += weight
    knowledge /= total_weight or 1
    metadata = 0.75 * content + 0.25 * knowledge

    co_likes = np.zeros(len(games))
    if cold_popularity[seed] == 0:
        return metadata, co_likes, metadata

    column = np.searchsorted(warm_items, seed)
    players_who_liked_seed = cold_likes_by_game[:, column].indices
    shared_likes = np.asarray(cold_likes[players_who_liked_seed].sum(axis=0)).ravel()
    denominator = np.sqrt(cold_popularity[warm_items] * cold_popularity[seed])
    co_likes[warm_items] = np.divide(shared_likes, denominator, out=np.zeros(len(warm_items)), where=denominator > 0)
    return 0.75 * co_likes + 0.25 * metadata, co_likes, metadata

In [9]:
def filter_games(players=None, max_minutes=None, max_weight=None):
    """Return a mask for games that satisfy the selected limits."""
    eligible = np.ones(len(games), dtype=bool)
    if players is not None:
        eligible &= ((games["minplayers"] <= players) & (players <= games["maxplayers"])).to_numpy()
    if max_minutes is not None:
        eligible &= (games["maxplaytime"] <= max_minutes).to_numpy()
    if max_weight is not None:
        eligible &= (games["averageweight"] <= max_weight).to_numpy()
    return eligible

In [10]:
def recommend_cold(seed_game_ids=(), players=None, max_minutes=None, max_weight=None, k=5):
    """Rank eligible games from liked game IDs, or from popularity alone."""
    seed_ids = [] if seed_game_ids is None else list(seed_game_ids)
    if len(seed_ids) > 3 or len(set(seed_ids)) != len(seed_ids):
        raise ValueError("Enter up to three distinct games.")
    if not isinstance(k, (int, np.integer)) or k <= 0:
        raise ValueError("k must be a positive integer.")
    seeds = game_ids.get_indexer(seed_ids)
    if (seeds < 0).any():
        raise ValueError("Choose games from the catalog.")

    eligible = filter_games(players, max_minutes, max_weight)
    eligible[seeds] = False

    if len(seeds):
        seed_scores = [score_liked_game(seed) for seed in seeds]
        blends, co_likes, metadata = zip(*seed_scores)
        scores = 0.80 * np.mean(blends, axis=0) + 0.20 * popularity_prior
        co_likes, metadata = np.array(co_likes), np.array(metadata)
    else:
        eligible &= cold_popularity > 0
        scores = cold_popularity

    candidates = np.flatnonzero(eligible)
    order = np.lexsort((game_ids[candidates], -cold_popularity[candidates], -scores[candidates]))
    ranked = candidates[order[:k]]
    result = games.loc[ranked, ["game_id", "title", *limits]].copy()
    result["score"] = scores[ranked]
    reasons = []
    for item in ranked:
        if len(seeds) and co_likes[:, item].max() > 0:
            titles = games.loc[seeds[co_likes[:, item] > 0], "title"]
            reason = f"Players who liked {', '.join(titles)} also liked this game."
        elif len(seeds) and metadata[:, item].max() > 0:
            seed = seeds[np.flatnonzero(metadata[:, item] > 0)[0]]
            reason = f"Shares description terms or game attributes with {games.at[seed, 'title']}."
        elif not len(seeds):
            reason = "Popular among training users"
        elif cold_popularity[item]:
            reason = "Popular with players in the training data."
        else:
            reason = "Eligible game from the catalog."
        reasons.append(reason)
    result["reason"] = reasons
    return result.reset_index(drop=True)

In [11]:
def recommend_active(user_id, players=None, max_minutes=None, max_weight=None, k=5):
    """Rank unrated games using a user's rating profile."""
    if not isinstance(user_id, (int, np.integer)) or not 0 <= user_id < n_users:
        raise ValueError("Choose an existing rating profile.")
    eligible = filter_games(players, max_minutes, max_weight)
    eligible[history[user_id].indices] = False
    candidates = warm_items[eligible[warm_items]]
    if not len(candidates):
        return games.loc[[], ["game_id", "title", *limits]].assign(reason="")
    scores = ease_hybrid_scores(user_id, candidates)[selected_ease]
    ranked = top_items(scores, candidates, k)
    result = games.loc[ranked, ["game_id", "title", *limits]].copy()

    # Explain the strongest positive contribution from a game this user liked.
    profile = ease_profiles[user_id]
    reasons = []
    for game in np.searchsorted(warm_items, ranked):
        contributions = profile.data * ease_weights[profile.indices, game]
        contributions[profile.data < (LIKE / 10) ** 3] = 0
        reason = "Recommended using your rating history."
        if np.any(contributions > 0):
            liked_game = warm_items[profile.indices[contributions.argmax()]]
            reason = f"Players' ratings link this to {games.at[liked_game, 'title']}, which you liked."
        reasons.append(reason)
    return result.assign(reason=reasons).reset_index(drop=True)

## 7. Try the recommender

Choose liked games, a rating profile, or popular games. Each result shows why it was recommended.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

mode = widgets.Dropdown(
    options=["Liked games", "Rating profile", "Popular games"],
    description="Mode",
)
seed_titles = widgets.Textarea(
    value="Pandemic",
    description="Liked games",
    rows=3,
    layout=widgets.Layout(width="100%", max_width="28rem"),
)
hint = widgets.HTML("Enter up to three titles or BoardGameGeek IDs, one per line.")
profile = widgets.Dropdown(options=test.index[:20].tolist(), description="User")
label = {"description_width": "initial"}
players = widgets.BoundedIntText(description="Players", min=0, max=100000, style=label)
minutes = widgets.BoundedIntText(description="Maximum minutes", min=0, max=100000, style=label)
complexity = widgets.BoundedFloatText(description="Max complexity (1–5)", min=0, max=5, style=label)
button = widgets.Button(description="Find games", button_style="primary")
output = widgets.Output()


def resolve_game(text):
    """Look up a game by ID or a unique exact title."""
    if text.isdecimal() and int(text) in game_ids:
        return int(text)
    matches = games.loc[games["title"].str.casefold().eq(text.casefold()), "game_id"]
    if len(matches) > 1:
        ids = ", ".join(str(game_id) for game_id in matches.head(5))
        raise ValueError(f"Use a BoardGameGeek ID for {text} ({ids}).")
    if matches.empty:
        raise ValueError(f"Game not found. Check the title or ID for {text}.")
    return int(matches.iloc[0])


def update_mode(change=None):
    seed_titles.layout.display = "" if mode.value == "Liked games" else "none"
    hint.layout.display = seed_titles.layout.display
    profile.layout.display = "" if mode.value == "Rating profile" else "none"
    output.clear_output()


def show_results(_):
    with output:
        clear_output(wait=True)
        filters = {
            "players": players.value or None,
            "max_minutes": minutes.value or None,
            "max_weight": complexity.value or None,
        }
        try:
            if 0 < complexity.value < 1:
                raise ValueError("Use complexity between 1 and 5, or 0 for any.")
            if mode.value == "Rating profile":
                result = recommend_active(profile.value, **filters)
            else:
                seeds = []
                if mode.value == "Liked games":
                    titles = [line.strip() for line in seed_titles.value.splitlines() if line.strip()]
                    if not 1 <= len(titles) <= 3:
                        raise ValueError("Enter one to three liked games.")
                    seeds = [resolve_game(title) for title in titles]
                result = recommend_cold(seeds, **filters)
            if result.empty:
                print("No games match these limits. Try increasing the time or complexity limit.")
            else:
                display(result[["title", *limits, "reason"]].round(2))
        except ValueError as error:
            print(error)


mode.observe(update_mode, names="value")
button.on_click(show_results)
update_mode()
display(widgets.VBox([
    mode, seed_titles, hint, profile,
    widgets.HTML("Use 0 for any limit."),
    players, minutes, complexity, button, output,
]))